In [5]:
# Import
import numpy as np
import pandas as pd
import sys, os

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Imports OK")

Imports OK


In [2]:
try:
    from google.colab import drive
    os.system('git clone https://github.com/ethanresek/luminal-classifiers')
    os.system('cd /content/luminal-classifiers && git pull')
    sys.path.append('/content/luminal-classifiers')
    from pre_process import preprocess, split_data
    CSV = '/content/luminal-classifiers/data/METABRIC_RNA_Mutation.csv'
    print('Working in Colab')

except ImportError:
    from pre_process import preprocess, split_data
    CSV = 'data/METABRIC_RNA_Mutation.csv'
    print('Working locally')

Working locally


In [4]:
DF = pd.read_csv(CSV, low_memory=False)
Y_OLD_NAME = 'pam50_+_claudin-low_subtype'
KEEP = list(DF.columns[31:520]) + [Y_OLD_NAME]

X, y = preprocess(DF, KEEP)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_SEED)

In [6]:
rf = RandomForestClassifier(class_weight='balanced')
p_dist = {
    'n_estimators': [100, 200, 500, 1000],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 'log2', 0.1, 0.25, 0.5],
    'max_leaf_nodes': [None, 50, 100, 200, 500],
    'min_impurity_decrease': [0.0, 0.001, 0.005, 0.01, 0.05],
    'max_samples': [0.5, 0.7, 0.8, 0.9, None],
    'criterion': ['gini', 'entropy', 'log_loss']
}

rand_search = RandomizedSearchCV(rf, param_distributions=p_dist, scoring='f1', n_iter=100, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), random_state=RANDOM_SEED)
rand_search.fit(X_train, y_train)
print(rand_search.best_params_)

KeyboardInterrupt: 

In [ ]:
best = rand_search.best_params_

grid_params = {}
for key, val in best.items():
    if isinstance(val, str) or val is None:
        grid_params[key] = [val]
    elif isinstance(val, int):
        step = max(1, round(0.1 * val))
        grid_params[key] = [max(1, val - step), val, val + step]
    elif isinstance(val, float):
        step = max(0.005, round(0.1 * val, 4))
        grid_params[key] = [max(0, round(val - step, 4)), val, val + step]

In [ ]:
post_rand_rf = rand_search.best_estimator_

grid_search = GridSearchCV